# Lab 1 — Agent vs Chatbot: the Organiser Planner Loop

**Worklet 2 — Course Project, Notes & Assignment Organizer**

| | |
|---|---|
| Phase | A · Foundations |
| Goal | Establish the agent loop *(clarify → plan → revise)* instead of a one-shot chatbot answer |
| Nodes activated | **Coordinator loop only** — "no specialist nodes yet" |
| Deliverable | A notebook agent that clarifies, plans and revises a course workspace outline |
| Human authority | **Course faculty** |

### The document's acceptance criterion, quoted

> **Accept:** Given a vague request it asks *at least two clarifying questions* and yields a coherent, *editable* plan.

### The document's pseudocode, quoted

```text
req  = clarify(user_request)         # ask until inputs complete
plan = propose_plan(req)             # outline for the organised course workspace
while user.wants_changes(plan):
    plan = revise(plan, user.feedback)
return plan                          # not final — a structured plan
```

### The document's pitfall, quoted

> **Pitfall to avoid.** Don't answer in one shot — without clarify/revise it's a chatbot, not an agent.

We will **demonstrate** that pitfall rather than just mention it: cell 8 runs the one-shot anti-pattern and puts it side by side with the agent.

## 0. Setup

Runs on the **scripted backend** — deterministic, offline, no API key needed. Add a `GOOGLE_API_KEY` later and flip `LLM_BACKEND=live` to use real Gemini; nothing else changes.

In [1]:
import sys
from pathlib import Path

# Work from the repo root so `src/` is importable without installing.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from course_organizer.config import get_settings
from course_organizer.logging_setup import setup_logging
from course_organizer.llm import ScriptedLLM, get_llm
from course_organizer.labs.lab01_planner_loop import (
    VAGUE_REQUEST,
    InteractiveFeedback,
    PlannerAgent,
    ScriptedFeedback,
    one_shot_chatbot,
    run_planner_loop,
)
from course_organizer.schemas import CourseWorkspacePlan, PlanStatus

settings = get_settings()
run_id = setup_logging(level=settings.log_level)
llm = ScriptedLLM(max_calls=200)   # deterministic, offline

print(f"run id          : {run_id}")
print(f"llm backend     : {settings.llm_backend}  (scripted = no API key needed)")
print(f"source adapter  : {settings.source_adapter}")
print(f"gemini model    : {settings.gemini_model}  (used only when backend=live)")


run id          : b1fe3fd830de
llm backend     : scripted  (scripted = no API key needed)
source adapter  : local
gemini model    : gemini-3.5-flash-lite  (used only when backend=live)


## 1. The concept in one picture

A **chatbot** takes text and returns text. An **agent** takes text, maintains *state*, and runs a **loop with its own termination condition**:

```text
CHATBOT   request ──► LLM ──► text.                      done.

AGENT     request ──► clarify ──► [still missing?] ──yes──┐
                       │                                 │
                       no                                └──► ask user ──┐
                       │                                                │
                       ▼                                                │
                   propose_plan ──► plan ◄────────── revise(plan, feedback)
                                          ▲                              │
                                          └──────────────────────────────┘
                                                    (wants changes?)
```

Three properties make that an **agent** rather than a chatbot, and each one is a separate test below:

1. **It detects its own missing inputs** and asks, rather than inventing them.
2. **It returns a typed object**, not prose, so later labs can consume it.
3. **It is revisable** — the user can change the plan and it actually changes.

## 2. The starting point: a deliberately vague request

This is the request the document's acceptance criterion is written against. It names **no course, no unit and no assignment** — so the agent has nothing concrete to plan from and is *forced* to ask.

In [2]:
print(VAGUE_REQUEST)
print()
print(f"characters: {len(VAGUE_REQUEST)}   words: {len(VAGUE_REQUEST.split())}")


I need to get my course organised but honestly I have not even started thinking about the details. Sort it out please.

characters: 118   words: 22


## 3. Run the loop

`ScriptedFeedback` stands in for the human: it answers questions and supplies revision turns from a list. The **agent code is identical** to the interactive case — only the `FeedbackProvider` differs. That is what makes the acceptance test deterministic.

In [3]:
provider = ScriptedFeedback(
    answers={
        "course_code": "CS201",
        "units": "units 1-3",
        "assignments": "all three",
        "as_of": "2026-10-15",
    },
    feedback_turns=[
        "also cover unit 5",
        "remove the step track project milestones",
        "add a step for peer review of drafts before publishing",
    ],
)

result = run_planner_loop(VAGUE_REQUEST, provider=provider, llm=llm, compare_one_shot=False)

print("LOOP TRACE (must be clarify -> propose_plan -> revise):")
for step in result.trace:
    print(f"   {step}")


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


... (5 outputs left out)

16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | planner_loop_complete


LOOP TRACE (must be clarify -> propose_plan -> revise):
   clarify
   propose_plan
   revise_loop
   revise:1
   revise:2
   revise:3


## 4. ACCEPTANCE PART 1 — did it ask at least two clarifying questions?

> *"Given a vague request it asks at least two clarifying questions..."*

In [4]:
print(f"QUESTIONS ASKED: {result.request.question_count}")
print("=" * 78)
for i, q in enumerate(result.questions_asked, 1):
    tag = "assumed" if q.assumed else "asked  "
    print(f"\nQ{i}  field={q.field}  [{tag}]")
    print(f"    question : {q.question}")
    print(f"    why      : {q.why_it_matters}")
    print(f"    answer   : {q.answer or '(none)'}")

print("\n" + "=" * 78)
print(f"PASS: asked {result.request.question_count} >= 2 questions")
assert result.request.question_count >= 2, "Lab 1 acceptance criterion failed"


QUESTIONS ASKED: 4

Q1  field=course_code  [asked  ]
    question : Which course is this for? (e.g. CS201 Data Structures, or DBMS402)
    why      : Every downstream step - notes, assignments, grades - is scoped to one course. Guessing it would scope the whole plan to the wrong syllabus.
    answer   : CS201

Q2  field=units  [asked  ]
    question : Which units or topics should be in scope? (e.g. units 1-3, or just 'Binary Search Trees')
    why      : The plan's coverage is derived from this. Too wide and the run blows its budget; too narrow and the workspace is incomplete.
    answer   : units 1-3

Q3  field=assignments  [asked  ]
    question : Which assignment(s) are in scope? (e.g. ASG-01, or 'all three', or 'none' for notes only)
    why      : Submission collection, similarity checking and grading all operate on assignments. With none in scope, only the notes stream is planned.
    answer   : all three

Q4  field=as_of  [asked  ]
    question : What date should I treat as 'now

## 5. The clarified request

A **typed object**, not a summary. This is the doc's Lab 1 *Output*: *"A clarified request"*. Lab 2 will pass this straight into typed tools — if it were prose, everything downstream would break.

In [5]:
print(result.request.summary())
print()
print("raw request preserved verbatim:")
print(f"  {result.request.raw_request!r}")
print("\nassumptions (never silent - doc 2.10 Lab 3 pitfall):")
for a in result.request.assumptions:
    print(f"  - {a}")

print("\nJSON form (what the rest of the pipeline sees):")
import json
print(json.dumps(result.request.model_dump(mode="json"), indent=2)[:900])


CS201 (Data Structures) | units: 1, 2, 3, 5 | assignments: ASG-01, ASG-02, ASG-03 | milestones: yes | approver: course_faculty

raw request preserved verbatim:
  'I need to get my course organised but honestly I have not even started thinking about the details. Sort it out please.'

assumptions (never silent - doc 2.10 Lab 3 pitfall):
  - include_milestones: assumed true (the doc lists milestone tracking as a standard work stream)

JSON form (what the rest of the pipeline sees):
{
  "raw_request": "I need to get my course organised but honestly I have not even started thinking about the details. Sort it out please.",
  "course_code": "CS201",
  "course_name": "Data Structures",
  "units": [
    "1",
    "2",
    "3",
    "5"
  ],
  "assignment_ids": [
    "ASG-01",
    "ASG-02",
    "ASG-03"
  ],
  "as_of": "2026-10-15T00:00:00Z",
  "include_milestones": true,
  "human_authority": "course_faculty",
  "questions_asked": [
    {
      "field": "course_code",
      "question": "Which cour

## 6. ACCEPTANCE PART 2 — a coherent, editable plan

> *"...and yields a coherent, **editable** plan."*

**Coherent** = the steps are dependency-ordered, nothing is missing, and the governance spine the document mandates (validate → human gate → publish) is present in the right order.

**Editable** = it is a `DRAFT`, and the revise loop below really changes it.

In [6]:
print(result.plan.outline())


PLAN PLAN-CS201-0001  (status: draft, approver: course_faculty)
Request: CS201 (Data Structures) | units: 1, 2, 3, 5 | assignments: ASG-01, ASG-02, ASG-03 | milestones: yes | approver: course_faculty

  S1. Ingest course sources  -> Coordinator
      intent: Load notes, assignment/rubric definitions and submission records through the single governed connector (doc 2.7).
      done when: all three sources are read via the connector, none directly
  S2. Reconcile notes repository  -> Notes Repository Agent  [after S1]
      intent: Keep one current, versioned set of materials organised by unit/topic for units 1, 2, 3.
      done when: every in-scope unit resolves to exactly one current version
      done when: no prior version is removed
  S3. Distribute assignments on schedule  -> Assignment Distribution Agent  [after S1]
      intent: Release the faculty-approved briefs and rubrics for ASG-01, ASG-02, ASG-03 and schedule deadline reminders.
      done when: only faculty-approved briefs

In [7]:
plan = result.plan
ordered = plan._topological_steps()   # raises on a dependency cycle

print("COHERENCE CHECKS")
print("=" * 78)
print(f"  status is DRAFT (editable, not final) : {plan.status is PlanStatus.DRAFT}")
print(f"  dependency-ordered, no cycles         : True (topological sort succeeded)")
print(f"  every step has acceptance criteria   : {all(s.acceptance_criteria for s in plan.steps)}")
print(f"  every step has an intent             : {all(s.intent.strip() for s in plan.steps)}")
print(f"  human authority named                : {plan.human_authority}")

names = [s.name.lower() for s in ordered]
gate_i = next(i for i, n in enumerate(names) if plan.human_authority in n)
pub_i = next(i for i, n in enumerate(names) if "publish" in n)
print(f"  human gate precedes publish          : {gate_i < pub_i}  (hard stop, doc 2.3)")

assert plan.status is PlanStatus.DRAFT
assert gate_i < pub_i


COHERENCE CHECKS
  status is DRAFT (editable, not final) : True
  dependency-ordered, no cycles         : True (topological sort succeeded)
  every step has acceptance criteria   : True
  every step has an intent             : True
  human authority named                : course_faculty
  human gate precedes publish          : True  (hard stop, doc 2.3)


## 7. The revise loop, step by step

The doc: `while user.wants_changes(plan): plan = revise(plan, user.feedback)`.

Watch the **plan hash change** and the **step count move** with each turn. A revision that only wrote to a log would not be a revision.

In [8]:
agent = PlannerAgent(llm=llm)
working_plan = agent.propose_plan(result.request)

feedback_turns = [
    "also cover unit 5",
    "remove the step track project milestones",
    "add a step for peer review of drafts before publishing",
    "set the deadline to 2026-12-01",
]

print(f"{'turn':<6}{'steps':<8}{'units':<14}{'as_of':<14}plan hash")
print("-" * 78)
print(f"{'start':<6}{len(working_plan.steps):<8}{str(working_plan.request.units):<14}"
      f"{str(working_plan.request.as_of.date()):<14}{working_plan.plan_sha256()[:16]}")

for turn, feedback in enumerate(feedback_turns, 1):
    working_plan = agent.revise(working_plan, feedback)
    changes = working_plan.revisions[-1].changes_made
    print(f"r{turn:<5}{len(working_plan.steps):<8}{str(working_plan.request.units):<14}"
          f"{str(working_plan.request.as_of.date()):<14}{working_plan.plan_sha256()[:16]}")
    print(f"       feedback: {feedback}")
    for change in changes:
        print(f"         - {change}")

print("\nfinal plan id:", working_plan.plan_id)
print("revisions recorded:", len(working_plan.revisions))


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_proposed


turn  steps   units         as_of         plan hash
------------------------------------------------------------------------------
start 11      ['1', '2', '3', '5']2026-10-15    67bcd7e50cdcd12a
16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


r1    11      ['1', '2', '3', '5']2026-10-15    67bcd7e50cdcd12a
       feedback: also cover unit 5
         - scope widened to units 1, 2, 3, 5 (was 1, 2, 3, 5)
16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


r2    10      ['1', '2', '3', '5']2026-10-15    0b5e6ac8b6688ac9
       feedback: remove the step track project milestones
         - removed 1 step(s) matching 'track project milestones'
16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


r3    11      ['1', '2', '3', '5']2026-10-15    fb175c51f29ff20d
       feedback: add a step for peer review of drafts before publishing
         - added step S12 'For peer review of drafts before publishing'
16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


r4    11      ['1', '2', '3', '5']2026-12-01    fb175c51f29ff20d
       feedback: set the deadline to 2026-12-01
         - as_of moved 2026-10-15 -> 2026-12-01

final plan id: PLAN-CS201-0001
revisions recorded: 4


## 8. The document's pitfall, demonstrated

> *"Don't answer in one shot — without clarify/revise it's a chatbot, not an agent."*

Here is the anti-pattern, run through the **same LLM backend**, so the only difference is the loop.

In [9]:
one_shot = one_shot_chatbot(llm, VAGUE_REQUEST)

print("=" * 78)
print("SIDE-BY-SIDE")
print("=" * 78)
print(f"{'property':<34}{'one-shot chatbot':<22}agent")
print("-" * 78)
print(f"{'returns':<34}{'prose string':<22}typed CourseWorkspacePlan")
print(f"{'clarifying questions':<34}{'0':<22}{result.request.question_count}")
print(f"{'revisable':<34}{'no':<22}{'yes (' + str(len(working_plan.revisions)) + ' revisions applied)'}")
print(f"{'consumable by Lab 2+':<34}{'no':<22}yes")
print(f"{'has a hard stop before publish':<34}{'no':<22}yes (course faculty gate)")
print("\n" + "-" * 78)
print("one-shot output (truncated):")
print("  " + str(one_shot)[:220])

assert isinstance(one_shot, str)
assert not isinstance(one_shot, CourseWorkspacePlan)
assert result.request.question_count >= 2


SIDE-BY-SIDE
property                          one-shot chatbot      agent
------------------------------------------------------------------------------
returns                           prose string          typed CourseWorkspacePlan
clarifying questions              0                     4
revisable                         no                    yes (4 revisions applied)
consumable by Lab 2+              no                    yes
has a hard stop before publish    no                    yes (course faculty gate)

------------------------------------------------------------------------------
one-shot output (truncated):
  I need to get my course organised but honestly I have not even started thinking about the details. Sort it out please.


## 9. Control case — a request that needs no questions

A good agent does not nag. Given a request that already names everything, it should ask **nothing**.

In [10]:
concrete = (
    "Organise CS201 Data Structures for units 1-3, assignments ASG-01 and "
    "ASG-02, treating 2026-10-15 as today."
)

control = run_planner_loop(concrete, llm=llm, compare_one_shot=False)

print(f"questions asked : {control.request.question_count}")
print(f"clarified as    : {control.request.summary()}")
print(f"plan steps      : {len(control.plan.steps)}")

assert control.request.question_count == 0, "the agent should not nag"


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_complete


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_proposed


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | planner_loop_complete


questions asked : 0
clarified as    : CS201 (Data Structures) | units: 1, 2, 3 | assignments: ASG-01, ASG-02 | milestones: yes | approver: course_faculty
plan steps      : 11


## 10. The loop is bounded

The document warns in Lab 7 that *"an unbounded regenerate loop can spin forever"*. The same discipline is applied to the revise loop from Lab 1 — and hitting the bound is **recorded and escalated**, never hidden.

In [11]:
endless = ScriptedFeedback(feedback_turns=[f"add a step called extra thing {i}" for i in range(50)])
bounded = run_planner_loop(VAGUE_REQUEST, provider=endless, llm=llm, compare_one_shot=False)

print(f"revisions applied : {bounded.revision_count}   (cap = {len(bounded.plan.revisions)} recorded)")
print("escalated to the human rather than looping forever:")
for q in bounded.plan.open_questions:
    print(f"  ? {q}")

assert bounded.revision_count <= 5


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | clarify_question_asked


16:15:10 INFO    [b1fe3fd830de] course_organizer.llm | llm_call


... (8 outputs left out)

16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | plan_revised


16:15:10 WARNING [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | Revise loop hit its bound of 5 revisions. Stopping so the run cannot spin forever (doc 2.10 Lab 7 pitfall).


16:15:10 INFO    [b1fe3fd830de] course_organizer.labs.lab01_planner_loop | planner_loop_complete


revisions applied : 5   (cap = 5 recorded)
escalated to the human rather than looping forever:
  ? Stopped after 5 revisions; a human should confirm the remaining differences.


## 11. Try it yourself (interactive)

Uncomment and run. This replaces the scripted provider with a real person, using the **same** `PlannerAgent`.

In [12]:
# interactive = run_planner_loop(
#     "get my DBMS course sorted, it's a mess",
#     provider=InteractiveFeedback(),
#     llm=llm,
#     compare_one_shot=False,
# )
# print(interactive.plan.outline())

print("(commented out - it waits on stdin)")


(commented out - it waits on stdin)


## 12. What Lab 1 proved, and what it deliberately did not do

The doc's Lab 1 row says: *"Nodes activated. Coordinator loop (no specialist nodes yet)."* So everything above is the **Coordinator alone**. Every specialist agent named in the plan is a *plan entry*, not a running node — those come alive in Lab 7.

In [13]:
print("Lab 1 - requirement coverage")
print("=" * 78)
print("  [x] Clarifies the request         -> PlannerAgent.clarify")
print("  [x] Proposes a typed plan        -> PlannerAgent.propose_plan")
print("  [x] Revises on feedback          -> PlannerAgent.revise")
print("  [x] Asks >=2 clarifying questions-> _build_clarified_request")
print("  [x] Plan is editable, not final  -> PlanStatus.DRAFT")
print("  [x] Pitfall avoided (not one-shot)-> one_shot_chatbot contrast")
print("  [x] Coordinator loop only        -> no specialist node executed")

print()
print(result.summary())


Lab 1 - requirement coverage
  [x] Clarifies the request         -> PlannerAgent.clarify
  [x] Proposes a typed plan        -> PlannerAgent.propose_plan
  [x] Revises on feedback          -> PlannerAgent.revise
  [x] Asks >=2 clarifying questions-> _build_clarified_request
  [x] Plan is editable, not final  -> PlanStatus.DRAFT
  [x] Pitfall avoided (not one-shot)-> one_shot_chatbot contrast
  [x] Coordinator loop only        -> no specialist node executed

clarifying questions asked : 4
plan id                   : PLAN-CS201-0001
plan status               : draft
plan steps                : 11
revisions applied         : 3
approver (human authority): course_faculty
plan hash                 : 35dab77bc9d76dfc...
